# Notebook 2 - EDA and Distribution
Aman Kumar - 251810700231
Alok Chauhan - 251810700318

EDA means exploratory data analysis. basically we are just
exploring the data through charts to find patterns.

aman already cleaned the data in notebook 1 so i am just
loading that cleaned file here

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data = pd.read_csv("outputs/spam_cleaned.csv")

spam = data[data["label"] == "spam"]
ham  = data[data["label"] == "ham"]

print("loaded", len(data), "messages")
print("spam:", len(spam), "| ham:", len(ham))

: 

first lets check the basic stats for message length
spam messages should be longer based on what we read

In [ ]:
# average length for spam vs ham
print("average char count:")
print(data.groupby("label")["char_count"].mean().round(1))

print("\nmedian char count:")
print(data.groupby("label")["char_count"].median())

Spam average is 137 chars and ham is only 70!!
spam messages are almost 2x longer

let me also check word count

In [ ]:
print("average word count:")
print(data.groupby("label")["word_count"].mean().round(1))

spam = 23 words average, ham = 14 words average

now let me check how often each feature appears in spam vs ham
this will tell us which features are good spam indicators

In [ ]:
cols = ["has_url", "has_number", "has_currency",
        "has_free", "has_call", "has_txt", "has_phone"]

for c in cols:
    sp = spam[c].mean() * 100
    hm = ham[c].mean() * 100
    print(f"{c:<20}  spam: {sp:.1f}%   ham: {hm:.1f}%")

has_phone is super strong - 73% of spam has a phone number
but only 0.1% of ham has one!!

has_url is also strong - 16.5% spam vs 0.3% ham

now making the charts. going to make 6 charts in one figure

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("SMS Spam - EDA Charts", fontsize=14, fontweight="bold")

# chart 1 - message length distribution
for label, color in [("spam", "#E74C3C"), ("ham", "#2ECC71")]:
    d = data[data["label"] == label]["char_count"]
    ax[0,0].hist(d, bins=50, alpha=0.6, color=color, label=label, density=True)

ax[0,0].axvline(spam["char_count"].median(), color="#E74C3C",
                linestyle="--", lw=2,
                label=f"spam median={spam['char_count'].median():.0f}")
ax[0,0].axvline(ham["char_count"].median(), color="#2ECC71",
                linestyle="--", lw=2,
                label=f"ham median={ham['char_count'].median():.0f}")
ax[0,0].set_title("Message Length")
ax[0,0].set_xlabel("characters")
ax[0,0].legend(fontsize=8)

# chart 2 - word count boxplot
bp = ax[0,1].boxplot([spam["word_count"], ham["word_count"]],
                      tick_labels=["spam","ham"], patch_artist=True)
bp["boxes"][0].set_facecolor("#E74C3C")
bp["boxes"][1].set_facecolor("#2ECC71")
ax[0,1].set_title("Word Count")
ax[0,1].set_ylabel("words")

# chart 3 - how many spam vs ham
counts = data["label"].value_counts()
bars = ax[0,2].bar(counts.index, counts.values,
                    color=["#2ECC71","#E74C3C"], width=0.5)
ax[0,2].set_title("Spam vs Ham Count")
for bar, val in zip(bars, counts.values):
    pct = val/len(data)*100
    ax[0,2].text(bar.get_x() + bar.get_width()/2,
                  bar.get_height() + 20,
                  f"{val}\n({pct:.1f}%)", ha="center")
ax[0,2].set_ylim(0, 5200)

# chart 4 - exclamation marks
for label, color in [("spam","#E74C3C"), ("ham","#2ECC71")]:
    d = data[data["label"] == label]["exclamation"].clip(0, 8)
    ax[1,0].hist(d, bins=9, alpha=0.6, color=color, label=label, density=True)
ax[1,0].set_title("Exclamation Marks")
ax[1,0].set_xlabel("count (max 8)")
ax[1,0].legend()

# chart 5 - feature prevalence
names  = ["URL", "Number", "Prize", "FREE", "CALL", "TXT"]
scols  = ["has_url","has_number","has_currency","has_free","has_call","has_txt"]
srates = [spam[c].mean()*100 for c in scols]
hrates = [ham[c].mean()*100  for c in scols]

x = np.arange(len(names))
w = 0.35
ax[1,1].bar(x-w/2, srates, w, color="#E74C3C", label="spam")
ax[1,1].bar(x+w/2, hrates, w, color="#2ECC71", label="ham")
ax[1,1].set_title("Feature % in Spam vs Ham")
ax[1,1].set_xticks(x)
ax[1,1].set_xticklabels(names)
ax[1,1].legend()

# chart 6 - CDF to find length threshold
for label, color in [("spam","#E74C3C"), ("ham","#2ECC71")]:
    vals = np.sort(data[data["label"]==label]["char_count"])
    cdf  = np.arange(1, len(vals)+1) / len(vals)
    ax[1,2].plot(vals, cdf, color=color, label=label, lw=2)
ax[1,2].axvline(100, color="gray", linestyle="--", label="100 chars")
ax[1,2].set_xlim(0, 400)
ax[1,2].set_title("CDF - Char Count")
ax[1,2].set_xlabel("characters")
ax[1,2].legend(fontsize=8)

plt.tight_layout()
plt.savefig("../outputs/02_eda_charts.png", dpi=120)
plt.show()
print("saved!")

the charts look good. some things i noticed:

1. spam messages are clearly longer - the histogram shows spam
   peaks around 150 chars while ham peaks around 30-50

2. messages with URLs are way more likely spam (57x more common)

3. from the CDF chart - almost all spam is between 100-160 chars
   this could be used as a simple rule

4. the exclamation marks chart shows spam uses more !!! but
   its not that different so probably not a strong signal alone

one thing i want to understand more - why do spam messages
have such consistent length? aman said its because spammers
try to fit everything in exactly one SMS (160 char limit)
that makes sense

next - Alok is doing notebook 3 (word frequency)